In [1]:
import re
import json
from collections import defaultdict
from typing import Dict, List, Set, Tuple
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# File paths
xml_file_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Semantic_graph/Full_InfOnto.xml"
output_dir = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations"

# Extract classes, properties, and individuals

In [3]:
class OntologyParser:
    """Unified parser for extracting classes, individuals, and properties from RDF/OWL ontology files."""
    
    def __init__(self):
        self.classes = set()
        self.individuals = defaultdict(set)  # class -> set of individuals
        self.properties = defaultdict(set)   # class -> set of properties
        self.uri_base = "http://www.cee.umd.edu/Energy/"
    
    def parse_uri(self, uri: str) -> Tuple[List[str], str, str]:
        """
        Parse a URI to extract classes, fragment, and type.
        
        Args:
            uri: The URI to parse
            
        Returns:
            Tuple of (classes_list, fragment, fragment_type)
            where fragment_type is 'individual', 'property', or 'class'
        """
        if not uri.startswith(self.uri_base):
            return [], "", "unknown"
        
        # Remove base URI
        remainder = uri[len(self.uri_base):]
        
        # Split by '#' to separate path and fragment
        if '#' in remainder:
            path_part, fragment = remainder.split('#', 1)
            
            # Extract classes from path (everything between '/' and before '#')
            classes = [cls for cls in path_part.split('/') if cls]
            
            # Determine fragment type - FIXED: Check for 'has' prefix first
            if fragment.startswith('has'):
                fragment_type = 'property'
            elif fragment and classes:  # Has classes and a fragment -> individual
                fragment_type = 'individual'
            else:
                fragment_type = 'class'
                
        else:
            # No fragment, treat entire path as classes
            path_components = [cls for cls in remainder.split('/') if cls and not remainder.endswith('/')]
            
            # Check if any component starts with 'has' - if so, it's a property
            if path_components and path_components[-1].startswith('has'):
                classes = path_components[:-1]  # All but the last component
                fragment = path_components[-1]  # The 'has...' component
                fragment_type = 'property'
            else:
                classes = path_components
                fragment = ""
                fragment_type = 'class'
        
        return classes, fragment, fragment_type
    
    def process_ontology_file(self, file_path: str) -> Dict:
        """
        Process an RDF/OWL file and extract all classes, individuals, and properties.
        
        Args:
            file_path: Path to the ontology file
            
        Returns:
            Dictionary with extracted ontology elements
        """
        print(f"🔍 Processing ontology file: {file_path}")
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            print(f"✓ File loaded successfully. Size: {len(content):,} characters")
        except Exception as e:
            print(f"❌ Error reading file: {e}")
            return {}
        
        # Extract all URIs from the ontology
        uri_pattern = re.compile(r'<(http://www\.cee\.umd\.edu/Energy/[^>]+)>')
        uris = uri_pattern.findall(content)
        
        print(f"✓ Found {len(set(uris)):,} unique URIs")
        
        # Process each unique URI
        for uri in set(uris):
            self._process_uri(uri)
        
        return self._compile_results()
    
    def _process_uri(self, uri: str):
        """Process a single URI and categorize it."""
        classes, fragment, fragment_type = self.parse_uri(uri)
        
        # Add classes
        for cls in classes:
            self.classes.add(cls)
        
        # Process fragment based on type
        if fragment_type == 'individual' and classes:
            # Add individual to the most specific class (last in the path)
            primary_class = classes[-1] if classes else 'Unknown'
            self.individuals[primary_class].add(fragment)
        elif fragment_type == 'property' and classes:
            # Add property to the most specific class
            primary_class = classes[-1] if classes else 'Unknown'
            self.properties[primary_class].add(fragment)
    
    def _compile_results(self) -> Dict:
        """Compile all extracted information into a structured dictionary."""
        # Clean up SeaDepth individuals by removing trailing ')'
        cleaned_individuals = {}
        for cls, individuals in self.individuals.items():
            if cls == 'SeaDepth':
                # Remove trailing ')' from SeaDepth individuals
                cleaned_individuals[cls] = sorted([ind.rstrip(')') for ind in individuals])
            else:
                cleaned_individuals[cls] = sorted(list(individuals))
        
        return {
            'classes': sorted(list(self.classes)),
            'individuals': cleaned_individuals,
            'properties': {cls: sorted(list(properties)) 
                         for cls, properties in self.properties.items()},
            'summary': {
                'total_classes': len(self.classes),
                'total_individuals': sum(len(individuals) for individuals in cleaned_individuals.values()),
                'total_properties': sum(len(properties) for properties in self.properties.values())
            }
        }
    
    def print_summary(self, results: Dict):
        """Print a formatted summary of the ontology."""
        print("\n" + "="*60)
        print("📋 ONTOLOGY STRUCTURE SUMMARY")
        print("="*60)
        
        print(f"\n📊 STATISTICS:")
        print(f"   Classes: {results['summary']['total_classes']}")
        print(f"   Individuals: {results['summary']['total_individuals']}")
        print(f"   Properties: {results['summary']['total_properties']}")
        
        print(f"\n🏷️  CLASSES ({len(results['classes'])}):")
        for i, cls in enumerate(results['classes'], 1):
            print(f"   {i:3d}. {cls}")
        
        print(f"\n👥 INDIVIDUALS BY CLASS:")
        for cls, individuals in results['individuals'].items():
            if individuals:
                print(f"   📂 {cls} ({len(individuals)} individuals):")
                for individual in individuals[:5]:  # Show first 5
                    print(f"     • {individual}")
                if len(individuals) > 5:
                    print(f"     ... and {len(individuals) - 5} more")
                print()
        
        print(f"\n🔧 PROPERTIES BY CLASS:")
        for cls, properties in results['properties'].items():
            if properties:
                print(f"   📂 {cls} ({len(properties)} properties):")
                for prop in properties[:5]:  # Show first 5
                    print(f"     • {prop}")
                if len(properties) > 5:
                    print(f"     ... and {len(properties) - 5} more")
                print()
    
    def save_results(self, results: Dict, output_directory, json_filename: str = None):
        """Save extraction results to multiple formats."""
        # Convert string path to Path object if needed
        if isinstance(output_directory, str):
            output_directory = Path(output_directory)
        
        output_directory.mkdir(exist_ok=True)
        
        # Set default JSON filename if not provided
        if json_filename is None:
            json_filename = "ontology_structure.json"
        elif not json_filename.endswith('.json'):
            json_filename = f"{json_filename}.json"
        
        # Save complete results as JSON
        json_path = output_directory / json_filename
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"💾 Complete results saved to: {json_path}")
        
        # Generate base name for text files from JSON filename
        base_name = json_filename.replace('.json', '')
        
        # Save classes as text file
        classes_path = output_directory / f"{base_name}_classes.txt"
        with open(classes_path, "w", encoding='utf-8') as f:
            f.write("CLASSES EXTRACTED FROM ONTOLOGY\n")
            f.write("="*50 + "\n\n")
            for i, cls in enumerate(results['classes'], 1):
                f.write(f"{i:3d}. {cls}\n")
        
        # Save individuals as text file
        individuals_path = output_directory / f"{base_name}_individuals.txt"
        with open(individuals_path, "w", encoding='utf-8') as f:
            f.write("INDIVIDUALS EXTRACTED FROM ONTOLOGY\n")
            f.write("="*50 + "\n\n")
            for cls, individuals in results['individuals'].items():
                if individuals:
                    f.write(f"\n{cls.upper()} CLASS:\n")
                    f.write("-" * (len(cls) + 7) + "\n")
                    for i, individual in enumerate(individuals, 1):
                        f.write(f"  {i:3d}. {individual}\n")
        
        # Save properties as text file
        properties_path = output_directory / f"{base_name}_properties.txt"
        with open(properties_path, "w", encoding='utf-8') as f:
            f.write("PROPERTIES EXTRACTED FROM ONTOLOGY\n")
            f.write("="*50 + "\n\n")
            for cls, properties in results['properties'].items():
                if properties:
                    f.write(f"\n{cls.upper()} CLASS:\n")
                    f.write("-" * (len(cls) + 7) + "\n")
                    for i, prop in enumerate(properties, 1):
                        f.write(f"  {i:3d}. {prop}\n")
        
        print(f"💾 Text files saved to:")
        print(f"   Classes: {classes_path}")
        print(f"   Individuals: {individuals_path}")
        print(f"   Properties: {properties_path}")

# Test the URI parsing with example URIs
def test_uri_parsing():
    """Test the parser with example URIs."""
    parser = OntologyParser()
    
    test_uris = [
        "http://www.cee.umd.edu/Energy/MSP/Interconnection#OCS-A-0487",
        "http://www.cee.umd.edu/Energy/MSP/Interconnection#hasLeaseNumber",
        "http://www.cee.umd.edu/Energy/Turbine#Turbine93",
        "http://www.cee.umd.edu/Energy/WindFarm#hasWindResource",
        "http://www.cee.umd.edu/Energy/MSP/NOAA#NWR80"
    ]
    
    print("🧪 TESTING URI PARSING:")
    print("-" * 40)
    for uri in test_uris:
        classes, fragment, fragment_type = parser.parse_uri(uri)
        print(f"URI: {uri}")
        print(f"  Classes: {classes}")
        print(f"  Fragment: {fragment}")
        print(f"  Type: {fragment_type}")
        print()

print("✅ OntologyParser class defined successfully!")

✅ OntologyParser class defined successfully!


In [4]:
# Test URI parsing first
test_uri_parsing()

🧪 TESTING URI PARSING:
----------------------------------------
URI: http://www.cee.umd.edu/Energy/MSP/Interconnection#OCS-A-0487
  Classes: ['MSP', 'Interconnection']
  Fragment: OCS-A-0487
  Type: individual

URI: http://www.cee.umd.edu/Energy/MSP/Interconnection#hasLeaseNumber
  Classes: ['MSP', 'Interconnection']
  Fragment: hasLeaseNumber
  Type: property

URI: http://www.cee.umd.edu/Energy/Turbine#Turbine93
  Classes: ['Turbine']
  Fragment: Turbine93
  Type: individual

URI: http://www.cee.umd.edu/Energy/WindFarm#hasWindResource
  Classes: ['WindFarm']
  Fragment: hasWindResource
  Type: property

URI: http://www.cee.umd.edu/Energy/MSP/NOAA#NWR80
  Classes: ['MSP', 'NOAA']
  Fragment: NWR80
  Type: individual



In [5]:
# Create parser instance and process the ontology
parser = OntologyParser()

# Process the full ontology file
results = parser.process_ontology_file(xml_file_path)

# Print comprehensive summary
parser.print_summary(results)

# Save all results with custom filename (you can change this)
custom_json_name = "Summary_SemanticGraph"  # Change this to your desired filename
parser.save_results(results, output_dir, json_filename=custom_json_name)

print(f"\n🎉 Processing complete! Check the output directory for detailed results.")
print(f"📁 JSON saved as: {custom_json_name}.json")

🔍 Processing ontology file: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Semantic_graph/Full_InfOnto.xml
✓ File loaded successfully. Size: 2,453,841 characters
✓ Found 1,650 unique URIs

📋 ONTOLOGY STRUCTURE SUMMARY

📊 STATISTICS:
   Classes: 38
   Individuals: 1378
   Properties: 239

🏷️  CLASSES (38):
     1. Cable
     2. Coral
     3. Decommission
     4. Design
     5. ECC
     6. EFH
     7. Event
     8. ExportCable
     9. ExternalEvent
    10. FailureEvent
    11. Geospatial
    12. Hurricane
    13. Installation
    14. Interconnection
    15. Landing
    16. LifeCycle
    17. MSP
    18. MaintenanceEvent
    19. NOAA
    20. OCS
    21. OCSw
    22. OM
    23. OperationalEvent
    24. Planning
    25. PowerLine
    26. Regulation
    27. Restricted
    28. SeaDepth
    29. Stage
    30. Substation
    31. Task
    32. Time
    33. Turbine
    34. WeatherEvent
    35. WindFarm
    36. WindLease
    37. WindResource
    38. WindSpeed

👥 

# Extract key words

In [6]:
import re
from collections import Counter
from typing import Dict, List, Set

class KeywordExtractor:
    """Extract keywords from ontology structure by loading Summary_SemanticGraph.json."""
    
    def __init__(self):
        self.ontology_data = None
        self.extracted_keywords = set()
        
        # Define translation mappings
        self.efh_translations = {
            "ahms": "atlantic highly migratory species",
            "cfmc": "caribbean fishery management council",
            "gafmc": "gulf of mexico fishery management council", 
            "npfmc": "north pacific fishery management council",
            "pfmc": "pacific fishery management council",
            "phms": "pacific highly migratory species",
            "safmc": "south atlantic fishery management council",
            "wpfmc": "western pacific fishery management council"
        }
        
        self.general_translations = {
            "ecc": "export cable corridor",
            "efh": "essential fish habitat",
            "msp": "marine spatial planning",
            "ocsw": "outer continental shelf withdrawal",
            "ocs": "outer continental shelf"
        }
    
    def load_ontology_data(self, json_path: str = None):
        """
        Load the Summary_SemanticGraph.json file.
        
        Args:
            json_path: Path to the Summary_SemanticGraph.json file. 
                      If None, uses default path in the same directory as this notebook.
        """
        if json_path is None:
            # Default path - same directory as this notebook
            json_path = Path(output_dir) / "Summary_SemanticGraph.json"
        else:
            # Use provided path (can be string or Path object)
            json_path = Path(json_path)
        
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                self.ontology_data = json.load(f)
            print(f"✓ Loaded ontology data from: {json_path}")
            print(f"  Classes: {len(self.ontology_data.get('classes', []))}")
            print(f"  Individuals: {sum(len(individuals) for individuals in self.ontology_data.get('individuals', {}).values())}")
            print(f"  Properties: {sum(len(properties) for properties in self.ontology_data.get('properties', {}).values())}")
            return True
        except FileNotFoundError:
            print(f"❌ File not found: {json_path}")
            print(f"Please check the file path and make sure the file exists.")
            return False
        except json.JSONDecodeError as e:
            print(f"❌ Error parsing JSON file: {e}")
            return False
        except Exception as e:
            print(f"❌ Error loading ontology data: {e}")
            return False
    
    def clean_class_name(self, class_name: str) -> str:
        """Convert class names like 'LifeCycle' to 'life cycle'."""
        if not class_name:
            return ""
        
        # Handle camelCase and PascalCase by inserting spaces
        # Convert "LifeCycle" -> "Life Cycle"
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', class_name)
        
        # Handle sequences of capitals followed by lowercase
        # Convert "XMLHttpRequest" -> "XML Http Request"
        spaced = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', spaced)
        
        # Convert to lowercase and clean up extra spaces
        cleaned = ' '.join(spaced.lower().split())
        
        # Apply translations
        cleaned = self._apply_translations(cleaned)
        
        return cleaned
    
    def clean_property_name(self, property_name: str) -> str:
        """Convert property names like 'hasRotorDiameter' to 'rotor diameter'."""
        if not property_name:
            return ""
        
        # Remove 'has' prefix if present
        cleaned = property_name
        if cleaned.lower().startswith('has'):
            cleaned = cleaned[3:]  # Remove 'has'
        
        # Handle camelCase and PascalCase
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', cleaned)
        spaced = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', spaced)
        
        # Convert to lowercase and clean up
        cleaned = ' '.join(spaced.lower().split())
        
        # Clean up Unicode characters and special patterns
        cleaned = self._clean_unicode_and_patterns(cleaned)
        
        # Apply translations
        cleaned = self._apply_translations(cleaned)
        
        return cleaned
    
    def clean_individual_name(self, individual_name: str) -> str:
        """Convert individual names to clean keywords."""
        if not individual_name:
            return ""
        
        # Handle camelCase and PascalCase
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', individual_name)
        spaced = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', spaced)
        
        # Replace common separators with spaces
        spaced = re.sub(r'[_-]', ' ', spaced)
        
        # Remove numbers and special characters, keep only letters and spaces
        cleaned = re.sub(r'[^a-zA-Z\s]', ' ', spaced)
        
        # Convert to lowercase and clean up
        cleaned = ' '.join(cleaned.lower().split())
        
        # Apply translations
        cleaned = self._apply_translations(cleaned)
        
        return cleaned
    
    def _clean_unicode_and_patterns(self, text: str) -> str:
        """Clean up Unicode characters and fix specific patterns."""
        # Remove Unicode escape sequences like \\u003e
        text = re.sub(r'\\u[0-9a-fA-F]{4}', '', text)
        
        # Fix measurement patterns like "humidity2m" -> "humidity at 2m" or "speed10m" -> "speed at 10m"
        text = re.sub(r'([a-zA-Z])(\d+)m\b', r'\1 at \2m', text)
        
        # Fix "wind zone" (remove any trailing characters)
        text = re.sub(r'wind zone.*', 'wind zone', text)
        
        # Fix specific cases for "outer continental shelf name" -> "outer continental shelf"
        text = re.sub(r'outer continental shelf name', 'outer continental shelf', text)
        
        return text.strip()
    
    def _apply_translations(self, text: str) -> str:
        """Apply EFH and general translations to the text."""
        # Check EFH translations first (more specific)
        if text.lower() in self.efh_translations:
            return self.efh_translations[text.lower()]
        
        # Check general translations
        if text.lower() in self.general_translations:
            return self.general_translations[text.lower()]
        
        # Check if text contains any of the translation keys as whole words
        for key, value in self.general_translations.items():
            # Use word boundaries to match whole words only
            pattern = r'\b' + re.escape(key) + r'\b'
            if re.search(pattern, text.lower()):
                text = re.sub(pattern, value, text.lower())
                
        return text

    def extract_keywords_from_ontology(self) -> Dict[str, Set[str]]:
        """Extract all keywords from the loaded ontology."""
        if not self.ontology_data:
            print("❌ No ontology data loaded. Call load_ontology_data() first.")
            return {}
        
        keywords = {
            'classes': set(),
            'properties': set(), 
            'individuals': set(),
            'all_unique': set()
        }
        
        # Extract from classes
        print("🔍 Extracting keywords from classes...")
        for class_name in self.ontology_data.get('classes', []):
            cleaned = self.clean_class_name(class_name)
            if self.is_valid_keyword(cleaned):
                keywords['classes'].add(cleaned)
                keywords['all_unique'].add(cleaned)
        
        # Extract from properties
        print("🔍 Extracting keywords from properties...")
        for class_name, properties in self.ontology_data.get('properties', {}).items():
            for property_name in properties:
                cleaned = self.clean_property_name(property_name)
                if self.is_valid_keyword(cleaned):
                    keywords['properties'].add(cleaned)
                    keywords['all_unique'].add(cleaned)
        
        # Extract from individuals (sample to avoid too many)
        print("🔍 Extracting keywords from individuals...")
        for class_name, individuals in self.ontology_data.get('individuals', {}).items():
            # Take first 10 individuals from each class to avoid overwhelming
            for individual_name in individuals[:10]:
                cleaned = self.clean_individual_name(individual_name)
                if self.is_valid_keyword(cleaned):
                    keywords['individuals'].add(cleaned)
                    keywords['all_unique'].add(cleaned)
        
        return keywords
    
    def categorize_keywords_by_domain(self, keywords: Set[str]) -> Dict[str, Set[str]]:
        """Categorize keywords by domain/topic."""
        categories = {
            'wind_energy': set(),
            'structures': set(),
            'electrical': set(),
            'environmental': set(),
            'regulations': set(),
            'geographic': set(),
            'technical_specs': set(),
            'lifecycle': set(),
            'other': set()
        }
        
        # Define keyword patterns for each category
        patterns = {
            'wind_energy': ['wind', 'turbine', 'blade', 'rotor', 'nacelle', 'hub', 'tower', 'offshore', 'onshore'],
            'structures': ['foundation', 'structure', 'support', 'platform', 'monopile', 'jacket', 'floating'],
            'electrical': ['power', 'voltage', 'current', 'electrical', 'generator', 'transformer', 'grid', 'capacity'],
            'environmental': ['marine', 'wildlife', 'bird', 'fish', 'habitat', 'ecosystem', 'environment', 'noise'],
            'regulations': ['regulation', 'permit', 'compliance', 'standard', 'authority', 'agency', 'restricted', 'prohibited'],
            'geographic': ['area', 'zone', 'region', 'depth', 'ocean', 'sea', 'coastal', 'state', 'federal'],
            'technical_specs': ['diameter', 'height', 'speed', 'mass', 'angle', 'factor', 'ratio', 'efficiency'],
            'lifecycle': ['design', 'installation', 'operation', 'maintenance', 'decommission', 'planning', 'construction']
        }
        
        for keyword in keywords:
            categorized = False
            for category, pattern_words in patterns.items():
                if any(pattern in keyword.lower() for pattern in pattern_words):
                    categories[category].add(keyword)
                    categorized = True
                    break
            
            if not categorized:
                categories['other'].add(keyword)
        
        return categories
    
    def save_keywords(self, keywords: Dict[str, Set[str]], output_path: str = None):
        """Save extracted keywords to files."""
        if output_path is None:
            output_path = Path(output_dir) / "extracted_keywords"
        
        # Convert sets to sorted lists for JSON serialization
        keywords_for_json = {
            category: sorted(list(keyword_set)) 
            for category, keyword_set in keywords.items()
        }
        
        # Save as JSON
        json_path = f"{output_path}.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(keywords_for_json, f, indent=2, ensure_ascii=False)
        print(f"💾 Keywords saved to: {json_path}")
        
        # Save as text file for easy reading
        txt_path = f"{output_path}.txt"
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write("EXTRACTED KEYWORDS FROM ONTOLOGY\n")
            f.write("=" * 50 + "\n\n")
            
            # Write summary
            total_unique = len(keywords.get('all_unique', set()))
            f.write(f"📊 SUMMARY:\n")
            f.write(f"   Total unique keywords: {total_unique}\n")
            f.write(f"   From classes: {len(keywords.get('classes', set()))}\n")
            f.write(f"   From properties: {len(keywords.get('properties', set()))}\n")
            f.write(f"   From individuals: {len(keywords.get('individuals', set()))}\n\n")
            
            # Write keywords by source
            for category, keyword_set in keywords.items():
                if category != 'all_unique' and keyword_set:
                    f.write(f"📂 {category.upper().replace('_', ' ')} ({len(keyword_set)} keywords):\n")
                    for keyword in sorted(keyword_set):
                        f.write(f"   • {keyword}\n")
                    f.write("\n")
        
        print(f"💾 Keywords text file saved to: {txt_path}")
        
        # Also save categorized keywords
        if 'all_unique' in keywords:
            categorized = self.categorize_keywords_by_domain(keywords['all_unique'])
            categorized_path = f"{output_path}_categorized.txt"
            
            with open(categorized_path, 'w', encoding='utf-8') as f:
                f.write("KEYWORDS CATEGORIZED BY DOMAIN\n")
                f.write("=" * 50 + "\n\n")
                
                for category, keyword_set in categorized.items():
                    if keyword_set:
                        f.write(f"📂 {category.upper().replace('_', ' ')} ({len(keyword_set)} keywords):\n")
                        for keyword in sorted(keyword_set):
                            f.write(f"   • {keyword}\n")
                        f.write("\n")
            
            print(f"💾 Categorized keywords saved to: {categorized_path}")

    def print_summary(self, keywords: Dict[str, Set[str]]):
        """Print a summary of extracted keywords."""
        print("\n" + "="*60)
        print("🔑 KEYWORD EXTRACTION SUMMARY")
        print("="*60)
        
        total_unique = len(keywords.get('all_unique', set()))
        print(f"\n📊 STATISTICS:")
        print(f"   Total unique keywords: {total_unique}")
        print(f"   From classes: {len(keywords.get('classes', set()))}")
        print(f"   From properties: {len(keywords.get('properties', set()))}")
        print(f"   From individuals: {len(keywords.get('individuals', set()))}")
        
        # Show samples from each category
        for category, keyword_set in keywords.items():
            if category != 'all_unique' and keyword_set:
                sample_keywords = sorted(list(keyword_set))[:10]
                print(f"\n📂 Sample {category.upper()}:")
                for keyword in sample_keywords:
                    print(f"   • {keyword}")
                if len(keyword_set) > 10:
                    print(f"   ... and {len(keyword_set) - 10} more")
        
        # Show domain categorization
        if 'all_unique' in keywords:
            categorized = self.categorize_keywords_by_domain(keywords['all_unique'])
            print(f"\n📋 DOMAIN CATEGORIZATION:")
            for category, keyword_set in categorized.items():
                if keyword_set:
                    print(f"   {category.replace('_', ' ').title()}: {len(keyword_set)} keywords")
    
    def is_valid_keyword(self, keyword: str, min_length: int = 3) -> bool:
        """Check if a keyword is valid for extraction."""
        if not keyword or len(keyword) < min_length:
            return False
        
        # Skip single letters
        if len(keyword) == 1:
            return False
        
        # Skip if it's all numbers
        if keyword.isdigit():
            return False
        
        # Skip common stop words
        stop_words = {
            'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
            'by', 'from', 'up', 'about', 'into', 'through', 'during', 'before',
            'after', 'above', 'below', 'between', 'among', 'is', 'are', 'was', 'were',
            'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
            'would', 'could', 'should', 'may', 'might', 'must', 'shall', 'can', 'a', 'an'
        }
        
        return keyword.lower() not in stop_words

print("✅ KeywordExtractor class defined successfully!")

✅ KeywordExtractor class defined successfully!


In [7]:
# Extract keywords from ontology by specifying the JSON file path
extractor = KeywordExtractor()

# Load ontology data and extract keywords with improved cleaning
custom_json_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Summary_SemanticGraph.json"

success = extractor.load_ontology_data(custom_json_path)

if success:
    print("\n🔍 Extracting unique keywords from loaded ontology structure...")
    keywords = extractor.extract_keywords_from_ontology()
    
    # Print summary
    extractor.print_summary(keywords)
    
    # Save keywords to files (this will overwrite the previous version)
    extractor.save_keywords(keywords, str(Path(output_dir) / "ontology_keywords"))
    
    print(f"\n🎉 Keyword extraction complete!")
    print(f"📁 Files saved in: {output_dir}")
    print("\n✨ Additional fixes applied:")
    print("  • Fixed spacing in measurement terms (e.g., 'humidity2m' -> 'humidity at 2m')")
    print("  • Fixed 'outer continental shelf name' -> 'outer continental shelf'")
    print("  • Improved measurement pattern recognition")
    
else:
    print("❌ Could not load ontology data. Please check the file path.")

✓ Loaded ontology data from: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Summary_SemanticGraph.json
  Classes: 38
  Individuals: 1378
  Properties: 239

🔍 Extracting unique keywords from loaded ontology structure...
🔍 Extracting keywords from classes...
🔍 Extracting keywords from properties...
🔍 Extracting keywords from individuals...

🔑 KEYWORD EXTRACTION SUMMARY

📊 STATISTICS:
   Total unique keywords: 249
   From classes: 37
   From properties: 185
   From individuals: 50

📂 Sample CLASSES:
   • cable
   • coral
   • decommission
   • design
   • essential fish habitat
   • event
   • export cable
   • export cable corridor
   • external event
   • failure event
   ... and 27 more

📂 Sample PROPERTIES:
   • aep
   • aerodynamic aep
   • agency
   • airfoil series
   • area
   • area name
   • arrival time
   • aspect
   • availability
   • avg depth
   ... and 175 more

📂 Sample INDIVIDUALS:
   • akmnwr
   • asnms
   • atl

# Filter LLM outputs

In [8]:
import pandas as pd
from pathlib import Path
import json
from typing import Dict, List, Set, Tuple
from collections import defaultdict
import re

class ConstraintFilter:
    """Filter regulatory constraints from LLM output using ontology keywords."""
    
    def __init__(self):
        self.keywords = None
        self.df = None
        self.filtered_results = None
        
    def load_keywords(self, keywords_path: str = None):
        """Load ontology keywords from JSON file."""
        if keywords_path is None:
            keywords_path = Path(output_dir) / "ontology_keywords.json"
        
        try:
            with open(keywords_path, 'r', encoding='utf-8') as f:
                self.keywords = json.load(f)
            print(f"✓ Loaded {len(self.keywords.get('all_unique', []))} unique keywords")
            return True
        except Exception as e:
            print(f"❌ Error loading keywords: {e}")
            return False
    
    def load_csv_data(self, csv_path: str):
        """Load LLM output CSV data."""
        try:
            self.df = pd.read_csv(csv_path)
            print(f"✓ Loaded CSV with {len(self.df)} rows and {len(self.df.columns)} columns")
            print(f"  Columns: {list(self.df.columns)}")
            return True
        except Exception as e:
            print(f"❌ Error loading CSV: {e}")
            return False
    
    def has_numerical_values(self, row) -> Tuple[bool, str]:
        """Check if constraint has numerical values by checking the numerical_value column."""
        # Check the dedicated numerical_value column
        numerical_value = row.get('numerical_value', '')
        unit = row.get('unit', '')
        
        # Check if numerical_value column has content
        if pd.notna(numerical_value) and str(numerical_value).strip():
            # Combine with unit if available
            if pd.notna(unit) and str(unit).strip():
                value_with_unit = f"{numerical_value} {unit}"
            else:
                value_with_unit = str(numerical_value)
            return True, value_with_unit
        
        return False, ""
    
    def preprocess_text(self, text: str) -> str:
        """Clean and normalize text for comparison."""
        if pd.isna(text) or not isinstance(text, str):
            return ""
        
        # Convert to lowercase and normalize whitespace
        text = ' '.join(text.lower().split())
        
        # Remove common symbols and punctuation
        text = re.sub(r'[^\w\s]', ' ', text)
        
        # Handle common abbreviations and variations
        text = re.sub(r'\bwtg\b', 'wind turbine generator', text)
        text = re.sub(r'\boss\b', 'offshore substation', text)
        text = re.sub(r'\bmw\b', 'megawatt', text)
        text = re.sub(r'\bkv\b', 'kilovolt', text)
        text = re.sub(r'\bosc\b', 'outer continental shelf', text)
        text = re.sub(r'\bboem\b', 'bureau of ocean energy management', text)
        
        return text
    
    def calculate_keyword_score(self, text: str, keywords_set: Set[str]) -> Tuple[int, List[str]]:
        """Calculate relevance score based on keyword matches."""
        if not text or not keywords_set:
            return 0, []
        
        preprocessed_text = self.preprocess_text(text)
        matched_keywords = []
        
        for keyword in keywords_set:
            # Check for exact matches and partial matches
            if keyword in preprocessed_text:
                matched_keywords.append(keyword)
            elif any(word in preprocessed_text for word in keyword.split()):
                # Give partial credit for multi-word keywords
                matched_keywords.append(f"{keyword} (partial)")
        
        return len(matched_keywords), matched_keywords
    
    def calculate_relevance_score(self, row) -> Dict:
        """Calculate comprehensive relevance score for a constraint row."""
        if self.keywords is None:
            return {'total_score': 0, 'details': {}}
        
        # Text fields to analyze
        text_fields = {
            'requirement': row.get('requirement', ''),
            'scope': row.get('scope', ''),
            'constraint_type': row.get('constraint_type', ''),
            'related_domains': row.get('related_domains', ''),
            'source': row.get('source', '')
        }
        
        # Calculate scores for different keyword categories
        scores = {}
        all_matches = []
        
        for category, keyword_list in self.keywords.items():
            if category == 'all_unique':
                continue
                
            keyword_set = set(keyword_list)
            category_score = 0
            category_matches = []
            
            for field_name, field_text in text_fields.items():
                field_score, field_matches = self.calculate_keyword_score(field_text, keyword_set)
                category_score += field_score
                category_matches.extend(field_matches)
            
            scores[category] = {
                'score': category_score,
                'matches': list(set(category_matches))  # Remove duplicates
            }
            all_matches.extend(category_matches)
        
        # Calculate total score with weights
        weights = {
            'classes': 3.0,      # Core ontology concepts
            'properties': 2.0,    # Specific attributes
            'individuals': 1.5    # Specific instances
        }
        
        total_score = sum(
            scores.get(category, {}).get('score', 0) * weights.get(category, 1.0)
            for category in weights
        )
        
        return {
            'total_score': total_score,
            'category_scores': scores,
            'all_matches': list(set(all_matches)),
            'match_count': len(set(all_matches))
        }
    
    def filter_constraints(self, min_score: float = 1.0, require_numerical: bool = True) -> pd.DataFrame:
        """Filter constraints based on ontology relevance and numerical values."""
        if self.df is None or self.keywords is None:
            print("❌ Please load both CSV data and keywords first")
            return pd.DataFrame()
        
        print(f"🔍 Filtering {len(self.df)} constraints with minimum score: {min_score}")
        if require_numerical:
            print("📊 Requiring constraints to contain numerical values (checking numerical_value column)")
        
        # Calculate relevance scores and check for numerical values
        scores_data = []
        for idx, row in self.df.iterrows():
            score_info = self.calculate_relevance_score(row)
            has_numbers, numerical_value_text = self.has_numerical_values(row)
            
            scores_data.append({
                'index': idx,
                'total_score': score_info['total_score'],
                'match_count': score_info['match_count'],
                'matched_keywords': '; '.join(score_info['all_matches'][:10]),  # Top 10 matches
                'category_breakdown': str(score_info['category_scores']),
                'has_numerical_values': has_numbers,
                'extracted_numerical_value': numerical_value_text
            })
        
        # Create scores DataFrame
        scores_df = pd.DataFrame(scores_data)
        
        # Merge with original data
        self.filtered_results = self.df.copy()
        self.filtered_results['relevance_score'] = scores_df['total_score']
        self.filtered_results['match_count'] = scores_df['match_count']
        self.filtered_results['matched_keywords'] = scores_df['matched_keywords']
        self.filtered_results['category_breakdown'] = scores_df['category_breakdown']
        self.filtered_results['has_numerical_values'] = scores_df['has_numerical_values']
        self.filtered_results['extracted_numerical_value'] = scores_df['extracted_numerical_value']
        
        # Apply filters
        filtered_df = self.filtered_results[self.filtered_results['relevance_score'] >= min_score]
        
        if require_numerical:
            filtered_df = filtered_df[filtered_df['has_numerical_values'] == True]
        
        # Sort by relevance score
        filtered_df = filtered_df.sort_values('relevance_score', ascending=False)
        
        print(f"✓ Found {len(filtered_df)} constraints meeting all criteria")
        if len(filtered_df) > 0:
            print(f"  Score range: {filtered_df['relevance_score'].min():.1f} - {filtered_df['relevance_score'].max():.1f}")
            if require_numerical:
                numerical_count = filtered_df['has_numerical_values'].sum()
                print(f"  Constraints with numerical values: {numerical_count}")
        
        return filtered_df
    
    def analyze_filtering_results(self, filtered_df: pd.DataFrame):
        """Analyze and display filtering results."""
        if filtered_df.empty:
            print("❌ No filtered results to analyze")
            return
        
        print("\n" + "="*60)
        print("📊 FILTERING ANALYSIS")
        print("="*60)
        
        # Score distribution
        print(f"\n📈 SCORE DISTRIBUTION:")
        print(f"  Mean score: {filtered_df['relevance_score'].mean():.2f}")
        print(f"  Median score: {filtered_df['relevance_score'].median():.2f}")
        print(f"  Standard deviation: {filtered_df['relevance_score'].std():.2f}")
        
        # Numerical values analysis
        if 'has_numerical_values' in filtered_df.columns:
            numerical_count = filtered_df['has_numerical_values'].sum()
            print(f"\n📊 NUMERICAL VALUES:")
            print(f"  Constraints with numerical values: {numerical_count} ({numerical_count/len(filtered_df)*100:.1f}%)")
        
        # Top scoring constraints with numerical values
        print(f"\n🏆 TOP 5 HIGHEST SCORING CONSTRAINTS:")
        top_5 = filtered_df.head(5)
        for idx, row in top_5.iterrows():
            print(f"\n  #{idx} (Score: {row['relevance_score']:.1f})")
            print(f"    Type: {row.get('constraint_type', 'N/A')}")
            print(f"    Requirement: {str(row.get('requirement', ''))[:100]}...")
            print(f"    Numerical value: {str(row.get('extracted_numerical_value', 'N/A'))}")
            print(f"    Matched keywords: {str(row.get('matched_keywords', ''))[:80]}...")
        
        # Constraint type distribution
        if 'constraint_type' in filtered_df.columns:
            type_counts = filtered_df['constraint_type'].value_counts()
            print(f"\n📋 CONSTRAINT TYPE DISTRIBUTION:")
            for constraint_type, count in type_counts.head(10).items():
                print(f"  {constraint_type}: {count}")
        
        # Domain distribution
        if 'related_domains' in filtered_df.columns:
            # Extract domains from the related_domains column
            all_domains = []
            for domains_str in filtered_df['related_domains'].dropna():
                if isinstance(domains_str, str):
                    domains = [d.strip() for d in domains_str.split(';') if d.strip()]
                    all_domains.extend(domains)
            
            if all_domains:
                domain_counts = pd.Series(all_domains).value_counts()
                print(f"\n🌐 RELATED DOMAINS DISTRIBUTION:")
                for domain, count in domain_counts.head(10).items():
                    print(f"  {domain}: {count}")
    
    def save_filtered_results(self, filtered_df: pd.DataFrame, output_path: str = None):
        """Save filtered results to files."""
        if filtered_df.empty:
            print("❌ No filtered results to save")
            return
        
        if output_path is None:
            output_path = Path(output_dir) / "filtered_constraints"
        
        # Save as CSV
        csv_path = f"{output_path}.csv"
        filtered_df.to_csv(csv_path, index=False)
        print(f"💾 Filtered constraints saved to: {csv_path}")
        
        # Save summary as JSON
        summary = {
            'total_constraints': len(self.df) if self.df is not None else 0,
            'filtered_constraints': len(filtered_df),
            'filter_efficiency': len(filtered_df) / len(self.df) * 100 if self.df is not None and len(self.df) > 0 else 0,
            'constraints_with_numerical_values': int(filtered_df['has_numerical_values'].sum()) if 'has_numerical_values' in filtered_df.columns else 0,
            'score_statistics': {
                'mean': float(filtered_df['relevance_score'].mean()),
                'median': float(filtered_df['relevance_score'].median()),
                'std': float(filtered_df['relevance_score'].std()),
                'min': float(filtered_df['relevance_score'].min()),
                'max': float(filtered_df['relevance_score'].max())
            },
            'top_constraint_types': filtered_df['constraint_type'].value_counts().head(10).to_dict() if 'constraint_type' in filtered_df.columns else {},
            'filtering_date': pd.Timestamp.now().isoformat()
        }
        
        json_path = f"{output_path}_summary.json"
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, indent=2, ensure_ascii=False)
        print(f"💾 Summary saved to: {json_path}")
        
        # Save detailed analysis
        analysis_path = f"{output_path}_analysis.txt"
        with open(analysis_path, 'w', encoding='utf-8') as f:
            f.write("REGULATORY CONSTRAINTS FILTERING ANALYSIS\n")
            f.write("=" * 50 + "\n\n")
            
            f.write(f"📊 OVERVIEW:\n")
            f.write(f"  Original constraints: {len(self.df) if self.df is not None else 0}\n")
            f.write(f"  Filtered constraints: {len(filtered_df)}\n")
            f.write(f"  Filter efficiency: {summary['filter_efficiency']:.1f}%\n")
            f.write(f"  Constraints with numerical values: {summary['constraints_with_numerical_values']}\n\n")
            
            f.write(f"📈 SCORE STATISTICS:\n")
            for stat, value in summary['score_statistics'].items():
                f.write(f"  {stat.capitalize()}: {value:.2f}\n")
            f.write("\n")
            
            f.write(f"🏆 TOP 10 CONSTRAINTS BY RELEVANCE:\n")
            top_10 = filtered_df.head(10)
            for i, (idx, row) in enumerate(top_10.iterrows(), 1):
                f.write(f"\n  {i}. Score: {row['relevance_score']:.1f}\n")
                f.write(f"     Type: {row.get('constraint_type', 'N/A')}\n")
                f.write(f"     Scope: {str(row.get('scope', ''))[:100]}...\n")
                f.write(f"     Requirement: {str(row.get('requirement', ''))[:150]}...\n")
                f.write(f"     Numerical value: {str(row.get('extracted_numerical_value', 'N/A'))}\n")
                f.write(f"     Keywords: {str(row.get('matched_keywords', ''))[:100]}...\n")
        
        print(f"💾 Detailed analysis saved to: {analysis_path}")

print("✅ ConstraintFilter class defined successfully!")

✅ ConstraintFilter class defined successfully!


In [9]:
# Example usage of the ConstraintFilter
print("🚀 REGULATORY CONSTRAINTS FILTERING DEMO")
print("="*50)

# Initialize the filter
filter_system = ConstraintFilter()

# Load keywords (using the ontology keywords we extracted earlier)
keywords_success = filter_system.load_keywords()

if keywords_success:
    # Load the CSV data (update this path to your actual CSV file)
    csv_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results/combined_regulatory_constraints.csv"
    
    csv_success = filter_system.load_csv_data(csv_path)
    
    if csv_success:
        print(f"\n🔍 Starting filtering process...")
        
        # Apply filtering with different thresholds
        print(f"\n--- FILTERING WITH SCORE >= 3.0 ---")
        high_relevance = filter_system.filter_constraints(min_score=3.0)
        
        if not high_relevance.empty:
            # Analyze results
            filter_system.analyze_filtering_results(high_relevance)
            
            # Save results
            filter_system.save_filtered_results(high_relevance, 
                                               str(Path(output_dir) / "high_relevance_constraints"))
        
        # Try with lower threshold for comparison
        print(f"\n--- FILTERING WITH SCORE >= 1.5 ---")
        medium_relevance = filter_system.filter_constraints(min_score=1.5)
        
        if not medium_relevance.empty:
            filter_system.save_filtered_results(medium_relevance, 
                                               str(Path(output_dir) / "medium_relevance_constraints"))
        
        print(f"\n🎉 Filtering complete!")
        print(f"📁 Check {output_dir} for filtered results")
        
    else:
        print(f"❌ Failed to load CSV data")
        print(f"Please check the CSV file path: {csv_path}")
        
else:
    print(f"❌ Failed to load keywords")
    print(f"Please make sure the ontology_keywords.json file exists")

🚀 REGULATORY CONSTRAINTS FILTERING DEMO
✓ Loaded 249 unique keywords
✓ Loaded CSV with 29254 rows and 12 columns
  Columns: ['model_name', 'document_number', 'type_of_wind_farm', 'chunk_number', 'document_id', 'constraint_type', 'requirement', 'scope', 'numerical_value', 'unit', 'source', 'related_domains']

🔍 Starting filtering process...

--- FILTERING WITH SCORE >= 3.0 ---
🔍 Filtering 29254 constraints with minimum score: 3.0
📊 Requiring constraints to contain numerical values (checking numerical_value column)
✓ Found 7396 constraints meeting all criteria
  Score range: 8.5 - 423.5
  Constraints with numerical values: 7396

📊 FILTERING ANALYSIS

📈 SCORE DISTRIBUTION:
  Mean score: 129.07
  Median score: 119.50
  Standard deviation: 57.46

📊 NUMERICAL VALUES:
  Constraints with numerical values: 7396 (100.0%)

🏆 TOP 5 HIGHEST SCORING CONSTRAINTS:

  #16229 (Score: 423.5)
    Type: Technical
    Requirement: Records documenting the make, model, maximum rated horsepower, engine displac

In [10]:
# Example usage of the ConstraintFilter with numerical value requirement
print("🚀 REGULATORY CONSTRAINTS FILTERING DEMO - WITH NUMERICAL VALUES")
print("="*65)

# Initialize the filter
filter_system = ConstraintFilter()

# Load keywords (using the ontology keywords we extracted earlier)
keywords_success = filter_system.load_keywords()

if keywords_success:
    # Load the CSV data (update this path to your actual CSV file)
    csv_path = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results/combined_regulatory_constraints.csv"
    
    csv_success = filter_system.load_csv_data(csv_path)
    
    if csv_success:
        print(f"\n🔍 Starting filtering process...")
        
        # Apply filtering with numerical value requirement
        print(f"\n--- FILTERING WITH SCORE >= 3.0 AND NUMERICAL VALUES ---")
        high_relevance_numerical = filter_system.filter_constraints(min_score=3.0, require_numerical=True)
        
        if not high_relevance_numerical.empty:
            # Analyze results
            filter_system.analyze_filtering_results(high_relevance_numerical)
            
            # Save results
            filter_system.save_filtered_results(high_relevance_numerical, 
                                               str(Path(output_dir) / "high_relevance_numerical_constraints"))
        else:
            print("❌ No constraints found meeting high relevance + numerical criteria")
        
        # Try with lower threshold
        print(f"\n--- FILTERING WITH SCORE >= 1.5 AND NUMERICAL VALUES ---")
        medium_relevance_numerical = filter_system.filter_constraints(min_score=1.5, require_numerical=True)
        
        if not medium_relevance_numerical.empty:
            filter_system.save_filtered_results(medium_relevance_numerical, 
                                               str(Path(output_dir) / "medium_relevance_numerical_constraints"))
            
            print(f"\n📊 SUMMARY OF NUMERICAL CONSTRAINTS:")
            print(f"  High relevance (≥3.0): {len(high_relevance_numerical)} constraints")
            print(f"  Medium relevance (≥1.5): {len(medium_relevance_numerical)} constraints")
        else:
            print("❌ No constraints found meeting medium relevance + numerical criteria")
        
        # For comparison, also filter without numerical requirement
        print(f"\n--- COMPARISON: FILTERING WITHOUT NUMERICAL REQUIREMENT ---")
        all_relevance = filter_system.filter_constraints(min_score=1.5, require_numerical=False)
        print(f"  Total constraints ≥1.5 score: {len(all_relevance)}")
        if 'has_numerical_values' in all_relevance.columns:
            numerical_portion = all_relevance['has_numerical_values'].sum()
            print(f"  With numerical values: {numerical_portion} ({numerical_portion/len(all_relevance)*100:.1f}%)")
        
        print(f"\n🎉 Filtering complete!")
        print(f"📁 Check {output_dir} for filtered results")
        print(f"\n💡 KEY INSIGHT: Only constraints with specific numerical values are retained")
        print(f"   These are more actionable for offshore wind project planning!")
        
    else:
        print(f"❌ Failed to load CSV data")
        print(f"Please check the CSV file path: {csv_path}")
        
else:
    print(f"❌ Failed to load keywords")
    print(f"Please make sure the ontology_keywords.json file exists")

🚀 REGULATORY CONSTRAINTS FILTERING DEMO - WITH NUMERICAL VALUES
✓ Loaded 249 unique keywords
✓ Loaded CSV with 29254 rows and 12 columns
  Columns: ['model_name', 'document_number', 'type_of_wind_farm', 'chunk_number', 'document_id', 'constraint_type', 'requirement', 'scope', 'numerical_value', 'unit', 'source', 'related_domains']

🔍 Starting filtering process...

--- FILTERING WITH SCORE >= 3.0 AND NUMERICAL VALUES ---
🔍 Filtering 29254 constraints with minimum score: 3.0
📊 Requiring constraints to contain numerical values (checking numerical_value column)
✓ Loaded CSV with 29254 rows and 12 columns
  Columns: ['model_name', 'document_number', 'type_of_wind_farm', 'chunk_number', 'document_id', 'constraint_type', 'requirement', 'scope', 'numerical_value', 'unit', 'source', 'related_domains']

🔍 Starting filtering process...

--- FILTERING WITH SCORE >= 3.0 AND NUMERICAL VALUES ---
🔍 Filtering 29254 constraints with minimum score: 3.0
📊 Requiring constraints to contain numerical values